# Lazy array kernels with Menudet

Use `@blosc2.menudet` to write lazy array kernels with arithmetic, branches and loops.
Write the calculation for one element; Blosc2 applies it across the input arrays.
Generic Python and Numba UDFs are covered separately in [User Defined Functions](03.lazyarray-udf.ipynb).

Menudet 0.1 is beta; API and file-format changes may occur with documented migration guidance.

In [1]:
import numpy as np

import blosc2


@blosc2.menudet(jit=False)
def adjusted(x, y):
    result = x + y
    if result < 0:
        result = 0
    return result


x = np.arange(6, dtype=np.float32)
y = np.full_like(x, -2)
lazy = adjusted(x, y)
np.testing.assert_array_equal(lazy[:], np.maximum(x + y, 0))
stored = lazy.compute()
stored[:]

array([0., 0., 0., 1., 2., 3.], dtype=float32)

`adjusted` returns a lazy calculation. The `if` applies separately to each element,
so negative sums become zero. Array inputs supply the element dtypes, and the
output dtype is inferred. Here the inputs and output are `float32`.

Use `lazy[:]` to read NumPy values, or `lazy.compute()` to materialize a Blosc2
array. `stored[:]` then reads the values from that materialized array.

`jit=False` selects native interpretation. For compiled execution, use
`@blosc2.menudet(jit=True)`. Neither mode executes the Python body, and unsupported
Python constructs raise an error rather than falling back to Python.

Continue with the [Menudet guide](../guides/menudet.rst) for scalar arguments,
backend selection, saving calculations and table integration. The
[syntax](../reference/dsl_syntax.md) and [numerical behavior](../reference/menudet_accuracy.md)
references describe the supported operations. For older DSL code or saved recipes,
see the [migration guide](../guides/migrating_to_menudet.rst).

## Saving a lazy calculation

To save the calculation rather than its computed values, first store its array
inputs on disk, then call `save()` on the lazy result. The recipe stores the
Menudet kernel and references to those inputs, so keep both input files available
when reopening it. Reading `reopened[:]` computes the values on demand.
The example uses `mode="w"` to overwrite any existing input files with these names.

In [2]:
x_disk = blosc2.asarray(x, urlpath="x.b2nd", mode="w")
y_disk = blosc2.asarray(y, urlpath="y.b2nd", mode="w")

lazy = adjusted(x_disk, y_disk)
lazy.save("adjusted.b2nd")

reopened = blosc2.open("adjusted.b2nd")
np.testing.assert_array_equal(reopened[:], np.maximum(x + y, 0))
reopened[:]

array([0., 0., 0., 1., 2., 3.], dtype=float32)

Once finished with this example, remove the recipe and its inputs:

In [3]:
blosc2.remove_urlpath("adjusted.b2nd")
blosc2.remove_urlpath("x.b2nd")
blosc2.remove_urlpath("y.b2nd")